In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, LongType, TimestampType, DecimalType

In [2]:
spark = (
    SparkSession
    .builder
    .appName('groupbyagg')
    .getOrCreate()
)

In [3]:
df = spark.read.csv("sales_info.csv", inferSchema=True, header=True)

In [4]:
df.show()

+-------+-------+-----+
|Company| Person|Sales|
+-------+-------+-----+
|   GOOG|    Sam|  200|
|   GOOG|Charlie|  120|
|   GOOG|  Frank|  340|
|   MFST|   Tina|  600|
|   MFST|    Amy|  124|
|   MFST|Vanessa|  243|
|     F8|   Carl|  870|
|     F8|  Sarah|  350|
|   APPL|   John|  250|
|   APPL|  Linda|  130|
|   APPL|   Mike|  750|
|   APPL|  Chris|  350|
+-------+-------+-----+



In [6]:
df.printSchema()

root
 |-- Company: string (nullable = true)
 |-- Person: string (nullable = true)
 |-- Sales: integer (nullable = true)



In [12]:
#df = df.withColumnRenamed('Sales', 'OldSales').withColumn('Sales', df.OldSales + 0.0).drop('OldSales')
df.show()

+-------+-------+-----+
|Company| Person|Sales|
+-------+-------+-----+
|   GOOG|    Sam|200.0|
|   GOOG|Charlie|120.0|
|   GOOG|  Frank|340.0|
|   MFST|   Tina|600.0|
|   MFST|    Amy|124.0|
|   MFST|Vanessa|243.0|
|     F8|   Carl|870.0|
|     F8|  Sarah|350.0|
|   APPL|   John|250.0|
|   APPL|  Linda|130.0|
|   APPL|   Mike|750.0|
|   APPL|  Chris|350.0|
+-------+-------+-----+



In [14]:
df.printSchema()

root
 |-- Company: string (nullable = true)
 |-- Person: string (nullable = true)
 |-- Sales: double (nullable = true)



In [17]:
# Группировка выполняется по колонке или списку колонок
df.groupBy("Company")

GroupedData[grouping expressions: [Company], value: [Company: string, Person: string ... 1 more field], type: GroupBy]

In [18]:
df.groupBy("Company").mean().show()

+-------+-----------------+
|Company|       avg(Sales)|
+-------+-----------------+
|   APPL|            370.0|
|     F8|            610.0|
|   GOOG|            220.0|
|   MFST|322.3333333333333|
+-------+-----------------+



In [19]:
# Подсчет элементов в группе (null не считается)
df.groupBy("Company").count().show()

+-------+-----+
|Company|count|
+-------+-----+
|   APPL|    4|
|     F8|    2|
|   GOOG|    3|
|   MFST|    3|
+-------+-----+



In [20]:
# Max
df.groupBy("Company").max().show()

+-------+----------+
|Company|max(Sales)|
+-------+----------+
|   APPL|     750.0|
|     F8|     870.0|
|   GOOG|     340.0|
|   MFST|     600.0|
+-------+----------+



In [21]:
# Min
df.groupBy("Company").min().show()

+-------+----------+
|Company|min(Sales)|
+-------+----------+
|   APPL|     130.0|
|     F8|     350.0|
|   GOOG|     120.0|
|   MFST|     124.0|
+-------+----------+



In [22]:
# Sum
df.groupBy("Company").sum().show()

+-------+----------+
|Company|sum(Sales)|
+-------+----------+
|   APPL|    1480.0|
|     F8|    1220.0|
|   GOOG|     660.0|
|   MFST|     967.0|
+-------+----------+



In [23]:
# Применение агрегата ко всей колонке Sales
# Этот способ позволит использовать и функции, написанные вами,
# если передать имя функции на позицию max
df.agg({'Sales':'max'}).show()

+----------+
|max(Sales)|
+----------+
|     870.0|
+----------+



In [24]:
# правильное применение группировок
grouped = df.groupBy("Company")

In [25]:
grouped.agg({'Sales':'max'}).show()

+-------+----------+
|Company|max(Sales)|
+-------+----------+
|   APPL|     750.0|
|     F8|     870.0|
|   GOOG|     340.0|
|   MFST|     600.0|
+-------+----------+



In [26]:
# Functions содержит множество базовых функций
from pyspark.sql.functions import countDistinct, avg, stddev

In [30]:
# Расчет оригинальных значений
df.select(countDistinct('Sales').alias('count Distinct')).show()

+--------------+
|count Distinct|
+--------------+
|            11|
+--------------+



In [39]:
from pyspark.sql import functions as F

df.select(
    F.format_number(avg('Sales'), 2).alias('avg Sales'),
).show()

+---------+
|avg Sales|
+---------+
|   360.58|
+---------+



In [40]:
df.select(stddev("Sales")).show()

+------------------+
|     stddev(Sales)|
+------------------+
|250.08742410799007|
+------------------+



In [45]:
sales_std = df.select(stddev("Sales").alias('std'))

In [44]:
df.printSchema()

root
 |-- Company: string (nullable = true)
 |-- Person: string (nullable = true)
 |-- Sales: double (nullable = true)



In [46]:
sales_std.show()

+------------------+
|               std|
+------------------+
|250.08742410799007|
+------------------+



In [51]:
sales_std.select(F.format_number('std', 2).alias('round( std )')).show()

+------------+
|round( std )|
+------------+
|      250.09|
+------------+



In [52]:
# OrderBy

df.orderBy("Sales").show()

+-------+-------+-----+
|Company| Person|Sales|
+-------+-------+-----+
|   GOOG|Charlie|120.0|
|   MFST|    Amy|124.0|
|   APPL|  Linda|130.0|
|   GOOG|    Sam|200.0|
|   MFST|Vanessa|243.0|
|   APPL|   John|250.0|
|   GOOG|  Frank|340.0|
|     F8|  Sarah|350.0|
|   APPL|  Chris|350.0|
|   MFST|   Tina|600.0|
|   APPL|   Mike|750.0|
|     F8|   Carl|870.0|
+-------+-------+-----+



In [53]:
# Descending
df.orderBy(df['Sales'].desc()).show()

+-------+-------+-----+
|Company| Person|Sales|
+-------+-------+-----+
|     F8|   Carl|870.0|
|   APPL|   Mike|750.0|
|   MFST|   Tina|600.0|
|     F8|  Sarah|350.0|
|   APPL|  Chris|350.0|
|   GOOG|  Frank|340.0|
|   APPL|   John|250.0|
|   MFST|Vanessa|243.0|
|   GOOG|    Sam|200.0|
|   APPL|  Linda|130.0|
|   MFST|    Amy|124.0|
|   GOOG|Charlie|120.0|
+-------+-------+-----+



In [54]:
df.orderBy(df['Sales'].asc()).show()

+-------+-------+-----+
|Company| Person|Sales|
+-------+-------+-----+
|   GOOG|Charlie|120.0|
|   MFST|    Amy|124.0|
|   APPL|  Linda|130.0|
|   GOOG|    Sam|200.0|
|   MFST|Vanessa|243.0|
|   APPL|   John|250.0|
|   GOOG|  Frank|340.0|
|     F8|  Sarah|350.0|
|   APPL|  Chris|350.0|
|   MFST|   Tina|600.0|
|   APPL|   Mike|750.0|
|     F8|   Carl|870.0|
+-------+-------+-----+

